# 🌍 Earthquake Data Analysis — MACSIN4A2225
**Team members:**
- ELyas MEZIANI
- Quan LUU
- Michael LI
- Ahmed JAIDANE
- Tahirou LAOUAN MAGAGI
- ....

**GitHub repository:** [https://github.com/latomate07/data-analysis-project](https://github.com/latomate07/data-analysis-project)

---
## 📖 About the dataset
This dataset contains records of **significant earthquakes** (magnitude ≥ 6.5) collected from the USGS (United States Geological Survey). Each row represents one seismic event and includes:
- **Temporal dimension:** `date_time` — when the earthquake occurred
- **Spatial dimension:** `latitude`, `longitude`, `country`, `continent` — where it happened
- **Analytical dimension:** `magnitude`, `depth`, `tsunami`, `alert`, `sig`, `cdi`, `mmi` — characteristics of the event

The goal is to understand global earthquake patterns: which regions are most at risk, how seismic activity evolves over time, and what spatial clusters of earthquakes exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Installation des librairies manquantes
!pip install dash mlxtend -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 52.7 MB/s eta 0:00:00


In [ ]:
# ─────────────────────────────────────────────
# Import all required libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


---
## Step 1 — Data Collection & Exploration 🔍

In [ ]:
# ─────────────────────────────────────────────
# Function: load_data
# Input:  file_path (str) — path to the CSV file
# Output: df (DataFrame)  — raw loaded dataframe
# ─────────────────────────────────────────────
def load_data(file_path):
    """Load the earthquake CSV dataset into a pandas DataFrame."""
    df = pd.read_csv(file_path)  # Read the CSV file
    print(f'✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


# ─────────────────────────────────────────────
# Function: explore_data
# Input:  df (DataFrame) — raw dataframe
# Output: prints exploration summary (no return value)
# ─────────────────────────────────────────────
def explore_data(df):
    """Summarise the dataset: types, missing values, and statistics."""

    print('\n--- 1. First 5 rows ---')
    display(df.head())

    print('\n--- 2. Column names, non-null counts, and data types ---')
    df.info()

    print(f'\n--- 3. Shape: {df.shape[0]} rows × {df.shape[1]} columns ---')

    print('\n--- 4. Missing values per column ---')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    missing_df = pd.DataFrame({'Missing count': missing, 'Missing %': missing_pct})
    display(missing_df[missing_df['Missing count'] > 0])  # Show only columns with missing data

    print('\n--- 5. Range of values for numerical variables ---')
    display(df.describe())

In [ ]:
# ─────────────────────────────────────────────
# Function: preprocess_data
# Input:  df (DataFrame) — raw dataframe
# Output: df (DataFrame) — cleaned dataframe ready for analysis
# ─────────────────────────────────────────────
def preprocess_data(df):
    """Clean and prepare the dataset for analysis."""

    df = df.copy()

    # Convert date_time to proper datetime format (format: DD-MM-YYYY HH:MM)
    df['date_time'] = pd.to_datetime(df['date_time'], format='%d-%m-%Y %H:%M', errors='coerce')

    # Extract useful temporal features from the date
    df['year']  = df['date_time'].dt.year
    df['month'] = df['date_time'].dt.month
    df['hour']  = df['date_time'].dt.hour

    # Fill missing 'alert' values with 'unknown' (367 missing = ~47% — too many to drop)
    df['alert'] = df['alert'].fillna('unknown')

    # Fill missing 'continent' and 'country' with 'Unknown'
    df['continent'] = df['continent'].fillna('Unknown')
    df['country']   = df['country'].fillna('Unknown')

    # Drop rows where lat/lon is missing (cannot do spatial analysis without coordinates)
    df = df.dropna(subset=['latitude', 'longitude'])

    print(f'✅ Preprocessing done. Shape after cleaning: {df.shape[0]} rows × {df.shape[1]} columns')
    return df

---
## Step 2 — Indicators & Queries 📊

### Query 1 — Grouping Query: Average magnitude & total events per continent

**What it represents:** For each continent, we compute the average earthquake magnitude and the total number of seismic events recorded. This tells us which regions of the world are most seismically active and most intense on average.

**How it is calculated:** We group the dataframe by `continent`, then apply `.mean()` on `magnitude` and `.count()` on any column (here `sig`) to get the event count.

**How to interpret it:** A continent with a high average magnitude AND many events is a major seismic hotspot. Oceania and Asia are expected to rank highly because they sit on the Pacific Ring of Fire.

In [ ]:
# ─────────────────────────────────────────────
# Function: query_groupby_continent
# Input:  df (DataFrame) — cleaned dataframe
# Output: result (DataFrame) — aggregated stats per continent
# ─────────────────────────────────────────────
def query_groupby_continent(df):
    """Compute average magnitude and total event count per continent."""

    result = (
        df[df['continent'] != 'Unknown']  # Exclude rows with unknown continent
        .groupby('continent')
        .agg(
            avg_magnitude  = ('magnitude', 'mean'),   # Average magnitude per continent
            total_events   = ('magnitude', 'count'),  # Total earthquakes per continent
            avg_depth      = ('depth', 'mean'),       # Average depth per continent
            tsunami_count  = ('tsunami', 'sum')       # Number of tsunami-triggering events
        )
        .round(2)
        .sort_values('total_events', ascending=False)  # Sort by most active first
        .reset_index()
    )

    print('--- Query 1: Stats per continent ---')
    display(result)

    # Visualisation: double bar chart
    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(
        go.Bar(x=result['continent'], y=result['total_events'],
               name='Total events', marker_color='steelblue'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=result['continent'], y=result['avg_magnitude'],
                   name='Avg magnitude', mode='lines+markers',
                   marker=dict(color='tomato', size=10), line=dict(width=2)),
        secondary_y=True
    )

    fig.update_layout(
        title='🌍 Earthquake events & average magnitude per continent',
        xaxis_title='Continent',
        legend=dict(x=0.75, y=0.95),
        template='plotly_white'
    )
    fig.update_yaxes(title_text='Total events', secondary_y=False)
    fig.update_yaxes(title_text='Average magnitude', secondary_y=True)
    fig.show()

    return result

---
### Query 2 — Frequent Pattern Mining: Association rules between alert level, tsunami, and magnitude category

**What it represents:** We use the **Apriori algorithm** to find frequent combinations (itemsets) of earthquake characteristics: the alert level (`green`, `yellow`, `orange`, `red`), whether a tsunami occurred, and the magnitude category (`moderate` 6.5–7.0, `strong` 7.0–8.0, `major` ≥ 8.0).

**How it is calculated:** Each earthquake is encoded as a binary transaction (1 = attribute present, 0 = absent). The Apriori algorithm finds item combinations that appear together frequently (support ≥ 5%), then derives association rules with confidence ≥ 60%.

**How to interpret it:** A rule like `{green_alert, moderate_magnitude} → {no_tsunami}` with high confidence means that when an earthquake is categorised as green alert and moderate magnitude, it almost never generates a tsunami. This is useful for early warning systems.

In [ ]:
# ─────────────────────────────────────────────
# Function: query_frequent_patterns
# Input:  df (DataFrame) — cleaned dataframe
# Output: rules (DataFrame) — association rules found
# ─────────────────────────────────────────────
def query_frequent_patterns(df):
    """Apply Apriori algorithm to find frequent patterns between alert, tsunami, and magnitude category."""

    df = df.copy()

    # Create magnitude categories (bins)
    df['mag_category'] = pd.cut(
        df['magnitude'],
        bins=[6.0, 7.0, 8.0, 10.0],
        labels=['moderate_mag', 'strong_mag', 'major_mag']
    )

    # Encode tsunami as a readable label
    df['tsunami_label'] = df['tsunami'].apply(lambda x: 'tsunami_yes' if x == 1 else 'tsunami_no')

    # Keep only rows with known alert (not 'unknown')
    df_known = df[df['alert'] != 'unknown'].copy()
    df_known['alert_label'] = 'alert_' + df_known['alert'].astype(str)

    # Build transactions: each earthquake = a list of its attributes
    transactions = df_known[['alert_label', 'tsunami_label', 'mag_category']].dropna()
    transactions_list = transactions.astype(str).values.tolist()

    # One-hot encode the transactions
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions_list)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)

    # Apply Apriori (min support = 5%)
    frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

    print(f'✅ Found {len(frequent_itemsets)} frequent itemsets (support ≥ 5%)')
    display(frequent_itemsets.sort_values('support', ascending=False).head(15))

    # Generate association rules (min confidence = 60%)
    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
    rules = rules.sort_values('lift', ascending=False)

    print(f'\n✅ Found {len(rules)} association rules (confidence ≥ 60%)')
    display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

    # Visualisation: scatter plot confidence vs support, sized by lift
    if len(rules) > 0:
        rules_plot = rules.copy()
        rules_plot['antecedents_str'] = rules_plot['antecedents'].apply(lambda x: ', '.join(list(x)))
        rules_plot['consequents_str'] = rules_plot['consequents'].apply(lambda x: ', '.join(list(x)))
        rules_plot['rule'] = rules_plot['antecedents_str'] + ' → ' + rules_plot['consequents_str']

        fig = px.scatter(
            rules_plot.head(20),
            x='support', y='confidence', size='lift', color='lift',
            hover_name='rule',
            color_continuous_scale='RdYlGn',
            title='🔗 Association rules: support vs confidence (size = lift)',
            labels={'support': 'Support', 'confidence': 'Confidence', 'lift': 'Lift'},
            template='plotly_white'
        )
        fig.show()

    return rules

---
### Query 3 — Temporal Analysis: Monthly earthquake frequency over time

**What it represents:** We analyse how the number of significant earthquakes evolves month by month over the full time range of the dataset. We also add a rolling average (3-month window) to smooth short-term noise and reveal underlying trends.

**How it is calculated:** The `date_time` column is parsed into datetime format, then we resample by month (`resample('ME')`) and count events. A 3-month rolling mean is applied on top.

**How to interpret it:** Spikes in the time series indicate periods of unusually intense seismic activity (e.g., aftershock sequences). The rolling average tells us whether activity is globally increasing, decreasing, or stable over time.

In [ ]:
# ─────────────────────────────────────────────
# Function: query_temporal
# Input:  df (DataFrame) — cleaned dataframe with parsed date_time
# Output: monthly (DataFrame) — monthly event counts
# ─────────────────────────────────────────────
def query_temporal(df):
    """Analyse monthly earthquake frequency and plot the time series with rolling average."""

    # Set date_time as index for resampling
    df_time = df.set_index('date_time').sort_index()

    # Count events per month
    monthly = df_time['magnitude'].resample('ME').count().rename('event_count')

    # Compute average magnitude per month (to overlay)
    monthly_avg_mag = df_time['magnitude'].resample('ME').mean().rename('avg_magnitude')

    # 3-month rolling average to smooth the curve
    monthly_rolling = monthly.rolling(window=3, center=True).mean().rename('rolling_3m')

    monthly_df = pd.concat([monthly, monthly_avg_mag, monthly_rolling], axis=1).reset_index()

    print(f'📅 Time range: {monthly_df["date_time"].min().date()} → {monthly_df["date_time"].max().date()}')
    print(f'📈 Peak month: {monthly_df.loc[monthly_df["event_count"].idxmax(), "date_time"].strftime("%B %Y")} '
          f'({monthly_df["event_count"].max()} events)')

    # Visualisation
    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(
        go.Bar(x=monthly_df['date_time'], y=monthly_df['event_count'],
               name='Monthly events', marker_color='royalblue', opacity=0.6),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=monthly_df['date_time'], y=monthly_df['rolling_3m'],
                   name='3-month rolling avg', line=dict(color='orangered', width=2)),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=monthly_df['date_time'], y=monthly_df['avg_magnitude'],
                   name='Avg magnitude', mode='lines',
                   line=dict(color='green', width=1.5, dash='dot')),
        secondary_y=True
    )

    fig.update_layout(
        title='📅 Monthly earthquake frequency over time',
        xaxis_title='Date',
        template='plotly_white',
        legend=dict(x=0.01, y=0.99)
    )
    fig.update_yaxes(title_text='Number of events', secondary_y=False)
    fig.update_yaxes(title_text='Average magnitude', secondary_y=True)
    fig.show()

    return monthly_df

---
### Query 4 — Spatial Clustering: Identifying seismic hotspots with DBSCAN

**What it represents:** We apply the **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) algorithm to the geographic coordinates (latitude, longitude) of all earthquakes. This reveals natural geographic clusters — i.e., the seismic hotspots of the world.

**How it is calculated:** DBSCAN groups points that are close to each other (within `epsilon` degrees) if at least `min_samples` earthquakes are in that neighbourhood. Points that belong to no cluster are labelled as noise (-1). We use `epsilon = 5.0` degrees and `min_samples = 5` earthquakes as parameters.

**How to interpret it:** Each cluster represents a geographically dense zone of seismic activity, corresponding to tectonic plate boundaries or fault lines. The noise points (-1) are isolated earthquakes in less active regions. A large number of clusters confirms the global distribution of seismic risk along the Ring of Fire and other known fault zones.

In [ ]:
# ─────────────────────────────────────────────
# Function: query_spatial_clustering
# Input:  df (DataFrame) — cleaned dataframe with latitude & longitude
#         epsilon (float) — DBSCAN neighbourhood radius in degrees (default 5.0)
#         min_samples (int)— DBSCAN minimum points per cluster (default 5)
# Output: df_clustered (DataFrame) — dataframe with an added 'cluster' column
# ─────────────────────────────────────────────
def query_spatial_clustering(df, epsilon=5.0, min_samples=5):
    """Apply DBSCAN spatial clustering on earthquake coordinates and visualise on a world map."""

    df_clustered = df.copy()

    # Extract coordinate matrix (lat, lon)
    coords = df_clustered[['latitude', 'longitude']].values

    # Run DBSCAN — no normalisation needed since lat/lon are already comparable
    db = DBSCAN(eps=epsilon, min_samples=min_samples, metric='haversine',
                algorithm='ball_tree').fit(np.radians(coords))  # Convert to radians for haversine

    df_clustered['cluster'] = db.labels_  # -1 = noise, 0,1,2,... = cluster id

    # Summary stats
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise    = list(db.labels_).count(-1)
    print(f'✅ DBSCAN found {n_clusters} clusters and {n_noise} noise points (isolated earthquakes)')

    # Cluster-level stats: size and average magnitude
    cluster_stats = (
        df_clustered[df_clustered['cluster'] >= 0]
        .groupby('cluster')
        .agg(cluster_size=('magnitude', 'count'), avg_mag=('magnitude', 'mean'))
        .sort_values('cluster_size', ascending=False)
        .head(10)
    )
    print('\nTop 10 largest clusters:')
    display(cluster_stats)

    # Visualisation: interactive scatter map
    df_plot = df_clustered.copy()
    df_plot['cluster_str'] = df_plot['cluster'].astype(str)  # String for discrete colour scale
    df_plot['point_size']  = df_plot['magnitude'] * 2        # Bigger points for stronger quakes

    fig = px.scatter_geo(
        df_plot,
        lat='latitude', lon='longitude',
        color='cluster_str',
        size='point_size',
        hover_data=['magnitude', 'country', 'depth', 'tsunami'],
        projection='natural earth',
        title=f'🗺️ Spatial clustering of earthquakes (DBSCAN — {n_clusters} clusters)',
        template='plotly_white'
    )
    fig.update_layout(legend_title='Cluster ID')
    fig.show()

    return df_clustered

---
## Step 3 — Visualization Dashboard 📊

The Dash dashboard below brings all four queries together in a single interactive interface. It includes:
- A world map of earthquake locations coloured by magnitude
- A bar chart of events per continent
- A time series of monthly activity
- A histogram of magnitude distribution

Run the cell and open `http://127.0.0.1:8050` in your browser.

In [ ]:
# ─────────────────────────────────────────────
# Function: build_dashboard
# Input:  df (DataFrame)           — cleaned dataframe
#         df_monthly (DataFrame)   — output of query_temporal()
#         df_continent (DataFrame) — output of query_groupby_continent()
# Output: launches a Dash web application (no return value)
# ─────────────────────────────────────────────
def build_dashboard(df, df_monthly, df_continent):
    """Build and launch an interactive Dash dashboard with all indicators."""

    app = dash.Dash(__name__)

    # ── Color mapping for alert levels
    alert_colors = {'green': 'green', 'yellow': 'gold', 'orange': 'orange', 'red': 'red', 'unknown': 'grey'}

    # ── Precompute figures

    # Fig 1: World map
    fig_map = px.scatter_geo(
        df, lat='latitude', lon='longitude',
        color='magnitude', size='magnitude',
        hover_name='country',
        hover_data=['depth', 'tsunami', 'alert', 'date_time'],
        color_continuous_scale='YlOrRd',
        projection='natural earth',
        title='🌍 World map of earthquakes by magnitude'
    )

    # Fig 2: Events per continent
    fig_continent = px.bar(
        df_continent, x='continent', y='total_events',
        color='avg_magnitude', color_continuous_scale='Blues',
        text='total_events',
        title='📊 Earthquake count per continent'
    )

    # Fig 3: Time series
    fig_time = go.Figure()
    fig_time.add_trace(go.Bar(
        x=df_monthly['date_time'], y=df_monthly['event_count'],
        name='Monthly events', marker_color='steelblue', opacity=0.7
    ))
    fig_time.add_trace(go.Scatter(
        x=df_monthly['date_time'], y=df_monthly['rolling_3m'],
        name='3-month rolling avg', line=dict(color='tomato', width=2)
    ))
    fig_time.update_layout(title='📅 Monthly earthquake frequency')

    # Fig 4: Magnitude distribution
    fig_hist = px.histogram(
        df, x='magnitude', nbins=30,
        color_discrete_sequence=['teal'],
        title='📈 Magnitude distribution'
    )

    # Fig 5: Tsunami rate per alert level
    tsunami_alert = (
        df[df['alert'] != 'unknown']
        .groupby('alert')['tsunami']
        .mean()
        .reset_index()
        .rename(columns={'tsunami': 'tsunami_rate'})
    )
    tsunami_alert['tsunami_rate_pct'] = (tsunami_alert['tsunami_rate'] * 100).round(1)
    fig_tsunami = px.bar(
        tsunami_alert, x='alert', y='tsunami_rate_pct',
        color='alert',
        color_discrete_map=alert_colors,
        text='tsunami_rate_pct',
        title='🌊 Tsunami rate (%) per alert level'
    )

    # ── Dashboard layout
    app.layout = html.Div([

        # Header
        html.Div([
            html.H1('🌍 Earthquake Data Analysis Dashboard', style={'color': 'white', 'margin': '0'}),
            html.P('Dataset: Significant Earthquakes (magnitude ≥ 6.5) — MACSIN4A2225',
                   style={'color': '#ccc', 'margin': '5px 0 0 0'}),
            html.P('Team: ELyas MEZIANI | Quan LUU | Michael LI | Ahmed JAIDANE | Tahirou ADAMOU',
                   style={'color': '#aaa', 'margin': '3px 0 0 0', 'fontSize': '13px'})
        ], style={'backgroundColor': '#1a1a2e', 'padding': '20px 30px',
                  'borderBottom': '3px solid #e94560'}),

        # KPI row
        html.Div([
            html.Div([html.H2(str(len(df))),      html.P('Total earthquakes')],  style={'textAlign':'center','flex':'1','padding':'15px','backgroundColor':'#16213e','color':'white','borderRadius':'8px','margin':'5px'}),
            html.Div([html.H2(f"{df['magnitude'].max():.1f}"), html.P('Max magnitude')], style={'textAlign':'center','flex':'1','padding':'15px','backgroundColor':'#e94560','color':'white','borderRadius':'8px','margin':'5px'}),
            html.Div([html.H2(str(df['tsunami'].sum())),  html.P('Tsunami events')],  style={'textAlign':'center','flex':'1','padding':'15px','backgroundColor':'#0f3460','color':'white','borderRadius':'8px','margin':'5px'}),
            html.Div([html.H2(str(df['country'].nunique())), html.P('Countries affected')], style={'textAlign':'center','flex':'1','padding':'15px','backgroundColor':'#533483','color':'white','borderRadius':'8px','margin':'5px'}),
        ], style={'display':'flex','padding':'15px 20px','gap':'5px'}),

        # Row 1: map + continent bar
        html.Div([
            html.Div([dcc.Graph(figure=fig_map)],       style={'flex':'2','padding':'5px'}),
            html.Div([dcc.Graph(figure=fig_continent)], style={'flex':'1','padding':'5px'}),
        ], style={'display':'flex'}),

        # Row 2: time series + histogram
        html.Div([
            html.Div([dcc.Graph(figure=fig_time)], style={'flex':'2','padding':'5px'}),
            html.Div([dcc.Graph(figure=fig_hist)], style={'flex':'1','padding':'5px'}),
        ], style={'display':'flex'}),

        # Row 3: tsunami rate
        html.Div([
            html.Div([dcc.Graph(figure=fig_tsunami)], style={'flex':'1','padding':'5px'}),
        ], style={'display':'flex'}),

    ], style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': '#f4f6f8'})

    print('🚀 Dashboard running at http://127.0.0.1:8050')
    app.run(debug=False, port=8050)

---
## Main Block 🚀

This is the entry point of the project. All functions are called in order from here.

In [ ]:
# ─────────────────────────────────────────────
# MAIN — entry point of the project
# Calls all functions in order
# ─────────────────────────────────────────────
if __name__ == '__main__':

    # ── Step 1: Data collection & exploration
    file_path = 'data/earthquake_data.csv'   # Path relative to the repo root
    df_raw  = load_data(file_path)               # Load the CSV
    explore_data(df_raw)                         # Print exploration summary
    df      = preprocess_data(df_raw)            # Clean & enrich

    # ── Step 2a: Groupby query (continent stats)
    df_continent = query_groupby_continent(df)

    # ── Step 2b: Frequent pattern mining (Apriori)
    rules = query_frequent_patterns(df)

    # ── Step 2c: Temporal query (monthly frequency)
    df_monthly = query_temporal(df)

    # ── Step 2d: Spatial clustering (DBSCAN)
    df_clustered = query_spatial_clustering(df, epsilon=5.0, min_samples=5)

    # ── Step 3: Launch the dashboard
    build_dashboard(df, df_monthly, df_continent)

✅ Dataset loaded: 782 rows × 19 columns

--- 1. First 5 rows ---


,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
0,"M 7.0 - 18 km SW of Malango, Solomon Islands",7.0,22-11-2022 02:03,8,7,green,1,768,us,117,0.509,17.0,mww,14.000,-9.7963,159.596,"Malango, Solomon Islands",Oceania,Solomon Islands
1,"M 6.9 - 204 km SW of Bengkulu, Indonesia",6.9,18-11-2022 13:37,4,4,green,0,735,us,99,2.229,34.0,mww,25.000,-4.9559,100.738,"Bengkulu, Indonesia",NaN,NaN
2,M 7.0 -,7.0,12-11-2022 07:09,3,3,green,1,755,us,147,3.125,18.0,mww,579.000,-20.0508,-178.346,NaN,Oceania,Fiji
3,"M 7.3 - 205 km ESE of Neiafu, Tonga",7.3,11-11-2022 10:48,5,5,green,1,833,us,149,1.865,21.0,mww,37.000,-19.2918,-172.129,"Neiafu, Tonga",NaN,NaN
4,M 6.6 -,6.6,09-11-2022 10:14,0,2,green,1,670,us,131,4.998,27.0,mww,624.464,-25.5948,178.278,NaN,NaN,NaN



--- 2. Column names, non-null counts, and data types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      782 non-null    object 
 1   magnitude  782 non-null    float64
 2   date_time  782 non-null    object 
 3   cdi        782 non-null    int64  
 4   mmi        782 non-null    int64  
 5   alert      415 non-null    object 
 6   tsunami    782 non-null    int64  
 7   sig        782 non-null    int64  
 8   net        782 non-null    object 
 9   nst        782 non-null    int64  
 10  dmin       782 non-null    float64
 11  gap        782 non-null    float64
 12  magType    782 non-null    object 
 13  depth      782 non-null    float64
 14  latitude   782 non-null    float64
 15  longitude  782 non-null    float64
 16  location   777 non-null    object 
 17  continent  206 non-null    object 
 18  country    484 non-null    objec

,Missing count,Missing %
alert,367,46.9
location,5,0.6
continent,576,73.7
country,298,38.1



--- 5. Range of values for numerical variables ---


...

,cluster_size,avg_mag
cluster,,
0,782,6.941125


🚀 Dashboard running at http://127.0.0.1:8050
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
